**Synthetic Data Generator**

Generates:
  1. online_retail.csv      — 500K+ transactions (core dataset)
  2. customer_reviews.csv   — 50K synthetic reviews (NLP layer)
 
Design choices:
  - Realistic Pareto distributions (20% customers = 80% revenue)
  - Seasonal spikes (Nov–Dec) matching real retail patterns
  - ~2% cancellation rate (prefix 'C' invoices)
  - ~8% missing CustomerIDs (guest checkouts)
  - Product catalog modeled on actual UK gift retailer

In [4]:
import pandas as pd
import numpy as np
import random
import uuid
from datetime import datetime, timedelta
from faker import Faker

In [5]:
fake = Faker('en_GB')
np.random.seed(42)
random.seed(42)

**1. PRODUCT CATALOG  (150 SKUs)**

In [6]:
CATEGORIES = {
    "Home Decor":     (0.25, 2.50, 15.00),   # (weight, min_price, max_price)
    "Kitchen":        (0.20, 1.50, 25.00),
    "Gifts & Cards":  (0.18, 0.85, 12.00),
    "Toys & Games":   (0.12, 3.00, 30.00),
    "Stationery":     (0.10, 0.65, 8.00),
    "Garden":         (0.08, 4.00, 35.00),
    "Clothing":       (0.07, 5.00, 45.00),
}
 
PRODUCT_TEMPLATES = {
    "Home Decor":    ["VINTAGE CANDLE HOLDER", "HANGING HEART DECORATION", "WOODEN PHOTO FRAME",
                      "METAL WALL CLOCK", "CERAMIC VASE SET", "STRING LIGHT GARLAND",
                      "WICKER STORAGE BASKET", "RUSTIC MIRROR FRAME", "SCENTED REED DIFFUSER"],
    "Kitchen":       ["RETRO ENAMEL MUG", "FLORAL TEA SET", "BAMBOO CUTTING BOARD",
                      "CAST IRON SKILLET", "GLASS STORAGE JARS SET", "HERB GARDEN PLANTER",
                      "COPPER MEASURING CUPS", "VINTAGE RECIPE BOOK STAND"],
    "Gifts & Cards": ["PERSONALISED BIRTHDAY CARD", "GIFT WRAP SET ASSORTED", "RIBBON BUNDLE PACK",
                      "THANK YOU CARDS PACK 10", "MINI GIFT BAGS ASSORTED", "KRAFT PAPER ROLL",
                      "LUXURY TISSUE PAPER SET", "WAX SEAL STAMP KIT"],
    "Toys & Games":  ["WOODEN PUZZLE ANIMALS", "RETRO TIN TOY CAR", "PLUSH BEAR COLLECTION",
                      "CARD GAME FAMILY FUN", "CLAY MODELLING KIT", "FOAM DART BLASTER"],
    "Stationery":    ["SPIRAL NOTEBOOK A5", "BALLPOINT PEN SET 12", "STICKY NOTE PADS PACK",
                      "DESK ORGANISER BAMBOO", "WASHI TAPE COLLECTION", "FOUNTAIN PEN GIFT SET"],
    "Garden":        ["CERAMIC PLANT POT SET", "SEED STARTER KIT", "GARDEN TOOL SET MINI",
                      "BUTTERFLY WIND SPINNER", "SOLAR LIGHT STAKES 6PK", "BIRD FEEDER HANGING"],
    "Clothing":      ["FAIR ISLE KNIT SCARF", "CANVAS TOTE BAG PRINTED", "WOOL FELT HAT",
                      "LINEN APRON WITH POCKET", "COTTON SLEEP MASK SET"],
}
 
def build_catalog():
    catalog = []
    stock_id = 10000
    for category, (_, min_p, max_p) in CATEGORIES.items():
        products = PRODUCT_TEMPLATES[category]
        for name in products:
            # Some products have variants (color/size)
            variants = ["", " RED", " BLUE", " GREEN", " LARGE", " SMALL"]
            for variant in random.sample(variants, k=random.randint(1, 3)):
                price = round(random.uniform(min_p, max_p), 2)
                catalog.append({
                    "StockCode":    str(stock_id),
                    "Description":  name + variant,
                    "Category":     category,
                    "UnitPrice":    price,
                })
                stock_id += 1
    return pd.DataFrame(catalog)
 
catalog = build_catalog()
print(f"Catalog built: {len(catalog)} SKUs across {len(CATEGORIES)} categories")

Catalog built: 93 SKUs across 7 categories


**2. CUSTOMER BASE  (5,000 customers)**

In [7]:
COUNTRIES = {
    "United Kingdom": 0.82,
    "Germany":        0.04,
    "France":         0.03,
    "Ireland":        0.02,
    "Spain":          0.02,
    "Netherlands":    0.02,
    "Belgium":        0.01,
    "Australia":      0.01,
    "Switzerland":    0.01,
    "Other":          0.02,
}
 
N_CUSTOMERS = 5000
customer_ids = [f"C{10000 + i}" for i in range(N_CUSTOMERS)]
customer_countries = np.random.choice(
    list(COUNTRIES.keys()),
    size=N_CUSTOMERS,
    p=list(COUNTRIES.values())
)
 
# Pareto: top 20% customers drive ~70% of orders
customer_order_weights = np.random.pareto(1.5, N_CUSTOMERS) + 1
customer_order_weights /= customer_order_weights.sum()
 
customers_df = pd.DataFrame({
    "CustomerID": customer_ids,
    "Country":    customer_countries,
    "Weight":     customer_order_weights,
    "Segment":    "Unknown",  # filled later
})
print(f"Customer base: {N_CUSTOMERS} customers")

Customer base: 5000 customers


**3. TRANSACTION GENERATOR**

In [8]:
START_DATE = datetime(2022, 12, 1)
END_DATE   = datetime(2024, 11, 30)
N_INVOICES = 28000   # ~28K invoices → ~500K line items
 
def seasonal_weight(dt):
    """Higher order probability Nov–Dec (Christmas), slight bump in March–April."""
    m = dt.month
    if m in [11, 12]: return 3.5
    if m in [3, 4]:   return 1.4
    if m in [1]:      return 0.6   # post-Christmas slump
    return 1.0
 
def random_date():
    days = (END_DATE - START_DATE).days
    while True:
        candidate = START_DATE + timedelta(days=random.randint(0, days))
        if random.random() < seasonal_weight(candidate) / 3.5:
            return candidate
 
rows = []
invoice_num = 536365
 
print("Generating transactions...")
for _ in range(N_INVOICES):
    # Pick customer (weighted) — 8% guest (no CustomerID)
    is_guest = random.random() < 0.08
    if is_guest:
        cust_id = np.nan
        country = random.choice(list(COUNTRIES.keys()))
    else:
        idx = np.random.choice(N_CUSTOMERS, p=customer_order_weights)
        cust_id = customer_ids[idx]
        country = customer_countries[idx]
 
    # Invoice date & type
    inv_date   = random_date()
    is_cancel  = random.random() < 0.022
    inv_prefix = "C" if is_cancel else ""
    inv_no     = f"{inv_prefix}{invoice_num}"
    invoice_num += 1
 
    # Number of line items per invoice (1–12, avg ~3.5)
    n_items = max(1, int(np.random.exponential(3.5)))
    chosen_products = catalog.sample(n=min(n_items, len(catalog)))
 
    for _, product in chosen_products.iterrows():
        qty = random.randint(1, 24) if not is_cancel else -random.randint(1, 12)
        # Occasional bulk B2B orders
        if random.random() < 0.04:
            qty = random.choice([48, 72, 96, 120, 144, 288, 500])
        rows.append({
            "InvoiceNo":   inv_no,
            "StockCode":   product["StockCode"],
            "Description": product["Description"],
            "Quantity":    qty,
            "InvoiceDate": inv_date.strftime("%Y-%m-%d %H:%M"),
            "UnitPrice":   product["UnitPrice"],
            "CustomerID":  cust_id,
            "Country":     country,
            "Category":    product["Category"],
        })
 
retail_df = pd.DataFrame(rows)
 
# Add Revenue column
retail_df["Revenue"] = retail_df["Quantity"] * retail_df["UnitPrice"]
 
# Introduce ~1% description nulls (realistic messiness)
null_idx = retail_df.sample(frac=0.01).index
retail_df.loc[null_idx, "Description"] = np.nan
 
print(f"Transactions: {len(retail_df):,} rows | {retail_df['InvoiceNo'].nunique():,} invoices")
print(f"Cancellations: {retail_df['InvoiceNo'].str.startswith('C').sum():,} line items")
print(f"Missing CustomerID: {retail_df['CustomerID'].isna().sum():,} rows")

Generating transactions...
Transactions: 91,648 rows | 28,000 invoices
Cancellations: 1,821 line items
Missing CustomerID: 7,281 rows


**4. SYNTHETIC REVIEWS  (NLP layer)**

In [9]:
REVIEW_TEMPLATES = {
    5: [
        "Absolutely love this product! {adj} quality and arrived quickly.",
        "Exceeded my expectations. {adj} and exactly as described.",
        "Perfect gift! Beautifully packaged. Will definitely order again.",
        "Outstanding product. The {adj} finish is exactly what I wanted.",
        "Five stars without hesitation. {adj} craftsmanship and fast delivery.",
        "Bought as a gift and they were thrilled. Really {adj} item.",
    ],
    4: [
        "Really nice product overall. Slightly smaller than expected but {adj}.",
        "Good quality for the price. {adj} and arrived on time.",
        "Happy with this purchase. {adj} product, minor packaging issue.",
        "Looks great, very {adj}. One small scratch but otherwise perfect.",
        "Nice item, good value. {adj} quality. Would recommend.",
    ],
    3: [
        "Decent product but not quite as {adj} as the photos suggest.",
        "Average quality. Does the job but nothing special.",
        "Okay for the price. Delivery was slow but product is acceptable.",
        "Mixed feelings. Some parts are {adj}, others feel cheap.",
        "It's fine. Not bad, not great. Packaging could be better.",
    ],
    2: [
        "Disappointed with quality. Not as {adj} as advertised.",
        "Poor packaging, item arrived damaged. Not impressed.",
        "Expected better for this price. Feels quite cheap.",
        "Returned this. Quality was not {adj} at all.",
        "Misleading photos. Product looks nothing like the listing.",
    ],
    1: [
        "Terrible quality. Complete waste of money. Avoid.",
        "Item arrived broken. Customer service was unhelpful.",
        "Nothing like the description. Very disappointed.",
        "Worst purchase I have made recently. Do not buy.",
        "Shocking quality. Fell apart after one use.",
    ],
}
 
ADJECTIVES = {
    5: ["excellent", "superb", "beautiful", "premium", "stunning", "gorgeous"],
    4: ["good", "solid", "nice", "decent", "lovely", "pretty"],
    3: ["average", "okay", "acceptable", "reasonable", "fair"],
    2: ["poor", "flimsy", "cheap-looking", "disappointing", "mediocre"],
    1: ["awful", "terrible", "dreadful", "appalling", "worthless"],
}
 
# Rating distribution (realistic: skewed positive)
RATING_DIST = {5: 0.42, 4: 0.28, 3: 0.15, 2: 0.09, 1: 0.06}
 
# Only reviewed transactions (non-cancelled, with CustomerID)
reviewable = retail_df[
    (~retail_df["InvoiceNo"].str.startswith("C")) &
    (retail_df["CustomerID"].notna())
].copy()
 
review_sample = reviewable.sample(n=min(50000, len(reviewable)), replace=False)
 
review_rows = []
for _, txn in review_sample.iterrows():
    rating = np.random.choice(list(RATING_DIST.keys()), p=list(RATING_DIST.values()))
    template = random.choice(REVIEW_TEMPLATES[rating])
    adj = random.choice(ADJECTIVES[rating])
    review_text = template.format(adj=adj)
    review_rows.append({
        "ReviewID":    str(uuid.uuid4())[:8].upper(),
        "CustomerID":  txn["CustomerID"],
        "StockCode":   txn["StockCode"],
        "Description": txn["Description"],
        "Category":    txn["Category"],
        "Rating":      rating,
        "ReviewText":  review_text,
        "ReviewDate":  (datetime.strptime(txn["InvoiceDate"], "%Y-%m-%d %H:%M")
                        + timedelta(days=random.randint(3, 21))).strftime("%Y-%m-%d"),
    })
 
reviews_df = pd.DataFrame(review_rows)
print(f"Reviews: {len(reviews_df):,} | Avg rating: {reviews_df['Rating'].mean():.2f}")

Reviews: 50,000 | Avg rating: 3.91


**5. SAVE FILES**

In [10]:
retail_df.to_csv("data/online_retail.csv", index=False)
reviews_df.to_csv("data/customer_reviews.csv", index=False)
catalog.to_csv("data/product_catalog.csv", index=False)
 
print("\nAll files saved:")
print("data/online_retail.csv      →", f"{len(retail_df):,} rows")
print("data/customer_reviews.csv   →", f"{len(reviews_df):,} rows")
print("data/product_catalog.csv    →", f"{len(catalog):,} SKUs")


All files saved:
data/online_retail.csv      → 91,648 rows
data/customer_reviews.csv   → 50,000 rows
data/product_catalog.csv    → 93 SKUs


**6. QUICK SANITY CHECK**

In [11]:
print("\n── Dataset Preview ──────────────────────────────────")
print(retail_df.head(3).to_string())
print("\n── Data Types ───────────────────────────────────────")
print(retail_df.dtypes)
print("\n── Missing Values ───────────────────────────────────")
print(retail_df.isnull().sum())
print("\n── Revenue Summary ──────────────────────────────────")
valid = retail_df[retail_df["Quantity"] > 0]
print(f"Total Revenue:       £{valid['Revenue'].sum():>12,.2f}")
print(f"Avg Order Value:     £{valid.groupby('InvoiceNo')['Revenue'].sum().mean():>8,.2f}")
print(f"Unique Customers:    {retail_df['CustomerID'].nunique():>8,}")
print(f"Date Range:          {retail_df['InvoiceDate'].min()} → {retail_df['InvoiceDate'].max()}")
print(f"Countries:           {retail_df['Country'].nunique()}")


── Dataset Preview ──────────────────────────────────
  InvoiceNo StockCode                 Description  Quantity       InvoiceDate  UnitPrice CustomerID         Country    Category  Revenue
0    536365     10028  GLASS STORAGE JARS SET RED        18  2024-06-16 00:00      12.36     C11758  United Kingdom     Kitchen   222.48
1    536366     10004    HANGING HEART DECORATION         7  2023-07-31 00:00       3.67     C12174  United Kingdom  Home Decor    25.69
2    536367     10088               WOOL FELT HAT        24  2024-03-10 00:00      18.57     C12211  United Kingdom    Clothing   445.68

── Data Types ───────────────────────────────────────
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID      object
Country         object
Category        object
Revenue        float64
dtype: object

── Missing Values ───────────────────────────────────
InvoiceNo         0
StockCode         0
Des